# Export Simulated WDB Light Curves for SCoPe Feature Generation

This notebook turns the simulated `lcurve-rs` injections into the two pieces SCoPe expects when running `generate_features(..., doSpecificIDs=True)`:

1. A source-list parquet with `ztf_id` and `coordinates` columns.
2. A `local_lightcurves` list shaped like Kowalski `ZTF_sources` documents.

The phrase "trick SCoPe" is basically right, but the reliable way to do it is through SCoPe's existing `local_lightcurves` hook rather than trying to upload fake sources to Kowalski. The simulated rows keep the original non-variable WD's sky position and metadata, while replacing the photometry with `sim_mag`/`sim_magerr` from the lcurve injection.

In [1]:
from pathlib import Path
import json
import pickle
import sys

import numpy as np
import pandas as pd

PROJECT_SRC = Path.cwd() / "src"
if PROJECT_SRC.exists() and str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

from wdb_ml.bootstrap import bootstrap_paths

paths = bootstrap_paths()
SCOPE_ML_ROOT = paths["scope_ml"]

EXPORT_DIR = Path("data/scope_simulated_wdb_queue")
LC_DIR = EXPORT_DIR / "lightcurves"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LC_DIR.mkdir(parents=True, exist_ok=True)


## Load Saved Simulation Batch

This notebook now expects `simulate_lcurve_wdbs.ipynb` to have saved `sim_meta.parquet` and `sim_lightcurves.pkl` under `data/simulated_wdb_batch/`.


In [2]:
SIM_BATCH_DIR = Path("data/simulated_wdb_batch")


def load_simulated_batch(input_dir=SIM_BATCH_DIR):
    input_dir = Path(input_dir)
    sim_meta_path = input_dir / "sim_meta.parquet"
    sim_lightcurves_path = input_dir / "sim_lightcurves.pkl"
    if not sim_meta_path.exists() or not sim_lightcurves_path.exists():
        raise FileNotFoundError(
            "Missing saved simulation batch. Run simulate_lcurve_wdbs.ipynb through "
            "the final save cell first."
        )
    sim_meta = pd.read_parquet(sim_meta_path)
    with open(sim_lightcurves_path, "rb") as f:
        sim_lightcurves = pickle.load(f)
    return sim_meta, sim_lightcurves


sim_meta, sim_lightcurves = load_simulated_batch()
print(f"Loaded {len(sim_meta)} simulated systems and {len(sim_lightcurves)} lightcurves")
sim_meta.head()


Loaded 10 simulated systems and 10 lightcurves


,period_days,q,iangle,r1,r2,t1,t2,phase0,amplitude_scale,blend_fraction,...,dec,field,ccd,quad,filter,n_obs,logg1,logg2,rv1,rv2
0,0.043460,1.413578,89.145222,0.029405,0.034861,37788.448474,6182.661789,0.375568,3.553778,0.051053,...,21.065998,561.0,16.0,4.0,1,191,0.0,0.0,0.029405,0.034859
1,0.018104,1.097666,80.771637,0.020625,0.030784,35011.493096,5010.667760,0.473817,5.151380,0.272105,...,28.331561,637.0,15.0,4.0,1,423,0.0,0.0,0.020625,0.030783
2,0.012457,1.015588,89.211772,0.018546,0.027215,14407.515023,12052.703071,0.288696,4.527430,0.435731,...,6.787609,489.0,15.0,4.0,2,465,0.0,0.0,0.018546,0.027214
3,0.481801,1.139194,81.477841,0.014967,0.038433,28032.781804,6514.963661,0.955039,5.967852,0.291841,...,28.240863,637.0,14.0,4.0,1,486,0.0,0.0,0.014967,0.038430
4,0.007802,0.493565,82.887037,0.010520,0.012704,41960.521525,7215.835544,0.497019,7.663701,0.295182,...,42.302462,699.0,12.0,1.0,1,289,0.0,0.0,0.010520,0.012704


## Build SCoPe-Compatible IDs and Coordinates

SCoPe's source-list parquet expects `coordinates.radec_geojson.coordinates` in the ZTF/Kowalski convention, where stored longitude is `ra - 180`. When `generate_features` reads it back, it adds `180` to recover RA.

In [3]:
def make_synthetic_ztf_id(index, base=9_000_000_000_000_000):
    """Return a large integer ID unlikely to collide with real ZTF light-curve IDs."""
    return int(base + index)


def infer_ra_dec(sim_lc, meta_row=None):
    """Find RA/Dec from light-curve columns or associated metadata."""
    for ra_col, dec_col in [("ra", "dec"), ("scope_ra", "scope_dec")]:
        if ra_col in sim_lc.columns and dec_col in sim_lc.columns:
            ra = sim_lc[ra_col].dropna()
            dec = sim_lc[dec_col].dropna()
            if len(ra) and len(dec):
                return float(ra.iloc[0]), float(dec.iloc[0])

    if meta_row is not None:
        for ra_col, dec_col in [("ra", "dec"), ("scope_ra", "scope_dec")]:
            if ra_col in meta_row.index and dec_col in meta_row.index:
                if pd.notna(meta_row[ra_col]) and pd.notna(meta_row[dec_col]):
                    return float(meta_row[ra_col]), float(meta_row[dec_col])

    raise KeyError("Could not infer RA/Dec. Add ra/dec to sim_lc or sim_meta.")


def make_source_list_row(ztf_id, ra, dec, fritz_name=None):
    row = {
        "ztf_id": int(ztf_id),
        "coordinates": {"radec_geojson": {"coordinates": [float(ra) - 180.0, float(dec)]}},
    }
    if fritz_name is not None:
        row["fritz_name"] = fritz_name
    return row


## Convert Simulated Photometry to Kowalski-Like Light-Curve Documents

`generate_features(local_lightcurves=...)` expects a list of dicts like:

```python
{
    "_id": ztf_id,
    "filter": 1 or 2 or 3,
    "data": [{"hjd": ..., "mag": ..., "magerr": ..., "catflags": ...}, ...]
}
```

We write each simulated light curve as a flat parquet for inspection and also build the in-memory dict list.

In [4]:
def sim_df_to_scope_lc_doc(sim_lc, ztf_id, filter_id=None, field=None, ccd=None, quad=None):
    df = sim_lc.copy()
    required = {"hjd", "sim_mag", "sim_magerr"}
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"sim_lc is missing required columns: {sorted(missing)}")

    if filter_id is None:
        if "filter" in df.columns and df["filter"].notna().any():
            filter_id = int(df["filter"].dropna().mode().iloc[0])
        else:
            filter_id = 2

    out = pd.DataFrame({
        "hjd": df["hjd"].astype(float),
        "mag": df["sim_mag"].astype(float),
        "magerr": df["sim_magerr"].astype(float),
        "catflags": 0,
        "programid": df["programid"].astype(int) if "programid" in df.columns else 1,
    }).dropna(subset=["hjd", "mag", "magerr"])
    out = out[out["magerr"] > 0].sort_values("hjd").reset_index(drop=True)

    doc = {
        "_id": int(ztf_id),
        "filter": int(filter_id),
        "data": out.to_dict("records"),
    }
    if field is not None:
        doc["field"] = int(field)
    if ccd is not None:
        doc["ccd"] = int(ccd)
    if quad is not None:
        doc["quad"] = int(quad)
    return doc, out


def export_scope_queue(sim_meta, sim_lightcurves, export_dir=EXPORT_DIR):
    export_dir = Path(export_dir)
    lc_dir = export_dir / "lightcurves"
    export_dir.mkdir(parents=True, exist_ok=True)
    lc_dir.mkdir(parents=True, exist_ok=True)

    meta_by_id = {}
    if sim_meta is not None and len(sim_meta):
        meta_by_id = {row["sim_id"]: row for _, row in sim_meta.iterrows() if "sim_id" in row.index}

    source_rows = []
    lc_docs = []
    manifest_rows = []

    for index, (sim_id, sim_lc) in enumerate(sim_lightcurves.items()):
        meta_row = meta_by_id.get(sim_id)
        ztf_id = make_synthetic_ztf_id(index)
        ra, dec = infer_ra_dec(sim_lc, meta_row)

        filter_id = None
        for col in ["filter", "fid"]:
            if col in sim_lc.columns and sim_lc[col].notna().any():
                filter_id = int(sim_lc[col].dropna().mode().iloc[0])
                break

        field = meta_row.get("field") if meta_row is not None and "field" in meta_row.index else None
        ccd = meta_row.get("ccd") if meta_row is not None and "ccd" in meta_row.index else None
        quad = meta_row.get("quad") if meta_row is not None and "quad" in meta_row.index else None

        lc_doc, flat_lc = sim_df_to_scope_lc_doc(sim_lc, ztf_id, filter_id=filter_id, field=field, ccd=ccd, quad=quad)
        flat_lc.insert(0, "ztf_id", ztf_id)
        flat_lc.insert(0, "sim_id", sim_id)
        flat_path = lc_dir / f"{sim_id}.parquet"
        flat_lc.to_parquet(flat_path, index=False)

        source_rows.append(make_source_list_row(ztf_id, ra, dec, fritz_name=f"SIM_{sim_id}"))
        lc_docs.append(lc_doc)
        manifest_rows.append({
            "sim_id": sim_id,
            "ztf_id": ztf_id,
            "ra": ra,
            "dec": dec,
            "filter": lc_doc["filter"],
            "n_obs": len(flat_lc),
            "lightcurve_path": str(flat_path),
        })

    source_list = pd.DataFrame(source_rows)
    manifest = pd.DataFrame(manifest_rows)

    source_list.to_parquet(export_dir / "source_list.parquet", index=False)
    manifest.to_parquet(export_dir / "manifest.parquet", index=False)
    with open(export_dir / "local_lightcurves.pkl", "wb") as f:
        pickle.dump(lc_docs, f)

    with open(export_dir / "README.json", "w") as f:
        json.dump({
            "source_list": "source_list.parquet",
            "manifest": "manifest.parquet",
            "local_lightcurves_pickle": "local_lightcurves.pkl",
            "lightcurve_dir": "lightcurves/",
            "note": "Use source_list.parquet as fg_dataset and pass local_lightcurves.pkl contents to generate_features(local_lightcurves=...).",
        }, f, indent=2)

    return source_list, lc_docs, manifest


## Export Queue Files

Run this after `sim_meta` and `sim_lightcurves` exist.

In [5]:
source_list, local_lightcurves, manifest = export_scope_queue(sim_meta, sim_lightcurves)
print(f"Wrote {len(source_list)} source rows and {len(local_lightcurves)} local lightcurves to {EXPORT_DIR}")
source_list.head(), manifest.head()


Wrote 10 source rows and 10 local lightcurves to data/scope_simulated_wdb_queue


(             ztf_id                                        coordinates  \
 0  9000000000000000  {'radec_geojson': {'coordinates': [-101.591936...   
 1  9000000000000001  {'radec_geojson': {'coordinates': [97.45492480...   
 2  9000000000000002  {'radec_geojson': {'coordinates': [114.4234296...   
 3  9000000000000003  {'radec_geojson': {'coordinates': [100.2667951...   
 4  9000000000000004  {'radec_geojson': {'coordinates': [-137.615894...   
 
       fritz_name  
 0  SIM_sim_00000  
 1  SIM_sim_00001  
 2  SIM_sim_00002  
 3  SIM_sim_00003  
 4  SIM_sim_00004  ,
       sim_id            ztf_id          ra        dec  filter  n_obs  \
 0  sim_00000  9000000000000000   78.408063  21.065998       1    191   
 1  sim_00001  9000000000000001  277.454925  28.331561       1    423   
 2  sim_00002  9000000000000002  294.423430   6.787609       2    465   
 3  sim_00003  9000000000000003  280.266795  28.240863       1    486   
 4  sim_00004  9000000000000004   42.384106  42.302462       1

## Optional: Run SCoPe Feature Generation Directly

This avoids remote light-curve queries by passing `local_lightcurves`. You still probably want `skipCloseSources=True` for simulated data because the source positions are inherited from already-selected WDs.

In [6]:
# from tools.generate_features import generate_features
#
# feature_df = generate_features(
#     doSpecificIDs=True,
#     fg_dataset=str(EXPORT_DIR / "source_list.parquet"),
#     local_lightcurves=local_lightcurves,
#     skipCloseSources=True,
#     doCPU=True,
#     period_algorithms=["ELS_ECE_EAOV"],
#     doNotSave=False,
#     dirname="generated_features_simulated_wdb",
#     filename="gen_features_simulated_wdb",
#     min_n_lc_points=30,
#     min_cadence_minutes=5.0,
# )
# feature_df.head()


## Optional: Reload an Exported Queue

Use this if you exported queue files in one session and want to run feature generation later.

In [7]:
def load_exported_scope_queue(export_dir=EXPORT_DIR):
    export_dir = Path(export_dir)
    source_list = pd.read_parquet(export_dir / "source_list.parquet")
    manifest = pd.read_parquet(export_dir / "manifest.parquet")
    with open(export_dir / "local_lightcurves.pkl", "rb") as f:
        local_lightcurves = pickle.load(f)
    return source_list, local_lightcurves, manifest


# source_list, local_lightcurves, manifest = load_exported_scope_queue()
